# AG2 Multi-Agent Research Assistant with Groq

[AG2](https://ag2.ai) (formerly AutoGen) is an open-source framework for building multi-agent AI applications. In this tutorial, two agents collaborate on a research task — one searches the web using DuckDuckGo, the other reviews the findings and asks follow-up questions.

**What you'll learn:**
- How to create AG2 agents powered by Groq's fast inference
- How to use AG2's built-in `DuckDuckGoSearchTool` for live web search
- How to orchestrate a multi-agent conversation with `initiate_chat`

## Install Dependencies

In [ ]:
!pip install -q "ag2[groq,duckduckgo]>=0.11.0"

## Configure Groq

Set your Groq API key. You can get one from [console.groq.com/keys](https://console.groq.com/keys).

In [ ]:
import os

from autogen import ConversableAgent, LLMConfig

llm_config = LLMConfig(
    {
        "api_type": "groq",
        # Llama 4 Scout handles tool calling reliably on Groq.
        # You can also try "qwen/qwen3-32b" or "llama-3.3-70b-versatile".
        "model": "meta-llama/llama-4-scout-17b-16e-instruct",
        "api_key": os.environ.get("GROQ_API_KEY"),
    }
)

## Set Up Web Search

AG2 includes a built-in `DuckDuckGoSearchTool` that wraps the [duckduckgo-search](https://pypi.org/project/duckduckgo-search/) package. It requires no API key and returns structured results (title, link, snippet).

The tool uses AG2's **registration pattern**: we register it with the Researcher agent for LLM use (so the Researcher decides when to search) and with the Reviewer agent for execution (so the Reviewer runs the actual search call).

In [ ]:
from autogen.tools.experimental import DuckDuckGoSearchTool

search_tool = DuckDuckGoSearchTool()

## Create Agents

We create two agents that will collaborate:

- **Researcher**: A research specialist that uses the DuckDuckGo search tool to gather information, then synthesizes findings into a structured summary.
- **Reviewer**: A critical reviewer that evaluates research quality. It asks follow-up questions if the research is incomplete, and responds with `TERMINATE` when satisfied.

In [ ]:
researcher = ConversableAgent(
    name="Researcher",
    system_message=(
        "You are a research specialist. Your job is to use the duckduckgo_search "
        "tool to find information on the topic you are given. Search multiple times "
        "if needed to cover different angles. After gathering enough information, "
        "synthesize your findings into a clear, structured summary with key takeaways."
    ),
    llm_config=llm_config,
    is_termination_msg=lambda msg: "TERMINATE" in (msg.get("content") or ""),
    human_input_mode="NEVER",
)

reviewer = ConversableAgent(
    name="Reviewer",
    system_message=(
        "You are a critical research reviewer. Evaluate the research provided to you "
        "for completeness, accuracy, and depth. If the research is missing important "
        "perspectives or needs more detail, ask specific follow-up questions or request "
        "additional searches. When you are satisfied that the research is thorough and "
        "well-supported, respond with TERMINATE."
    ),
    llm_config=llm_config,
    human_input_mode="NEVER",
)

# Researcher decides when to call the search tool
search_tool.register_for_llm(researcher)

# Reviewer executes the actual search
search_tool.register_for_execution(reviewer)

## Run the Research Conversation

The Reviewer kicks off the conversation by sending a research request to the Researcher. The two agents then chat back and forth — the Researcher searches the web and synthesizes findings, while the Reviewer evaluates and asks follow-ups.

`max_turns=6` prevents runaway conversations.

In [ ]:
result = reviewer.initiate_chat(
    researcher,
    message=(
        "Research the current state of AI inference hardware. "
        "Compare the approaches of at least two companies. "
        "Provide a summary with key findings."
    ),
    max_turns=6,
)

## Review the Results

The `ChatResult` object contains the full conversation history and a summary. Let's inspect what happened.

In [ ]:
print("=== Conversation Summary ===\n")
print(result.summary)

print("\n\n=== Full Chat History ===\n")
for msg in result.chat_history:
    speaker = msg.get("name", msg["role"])
    content = msg.get("content", "")
    if content:
        print(f"--- {speaker} ---")
        print(content[:500])
        if len(content) > 500:
            print("... (truncated)")
        print()

## Learn More

- [AG2 Documentation](https://docs.ag2.ai) — guides, API reference, and examples
- [AG2 Tool Use Tutorial](https://docs.ag2.ai/docs/tutorial/tool-use) — more on registering tools with agents
- [Groq API Documentation](https://console.groq.com/docs) — models, rate limits, and API reference